# 🏁 Proyecto: detector de discurso de odio · **ejemplo resuelto**
*Curso de NLP · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

Esta es la solución de un estudiante al proyecto [«🏁 detector de odio»](https://colab.research.google.com/github/psalgadm7/miatlas-cuadernos/blob/main/cuadernos/nlp-detector-de-odio.ipynb),
adaptada para publicarse: los datos se cargan desde Hugging Face, las claves van en Secretos y las rutas usan una
sola `CARPETA`. **Intenta el proyecto antes de mirarla.** Las respuestas son de su autor: no son las únicas
correctas.

**Los datos:** el conjunto de **Davidson et al. (2017)**, *Automated Hate Speech Detection and the Problem of
Offensive Language*: tweets en inglés recogidos con el léxico de Hatebase.org y clasificados a mano por varias
personas (CrowdFlower) en **0 · discurso de odio**, **1 · lenguaje ofensivo** y **2 · ninguno**. Se carga desde
Hugging Face (`tdavidson/hate_speech_offensive`) y lo dividimos en 80 % para entrenar y 20 % para probar.

> ⚠ **Contenido sensible:** son tweets reales con insultos, groserías y discurso de odio. Se usan para enseñar a
> detectarlos.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-NLP")
else:
    CARPETA = Path("MiAtlas-NLP")
CARPETA.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)

In [ ]:
%pip install -q transformers datasets evaluate accelerate

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

crudo = load_dataset("tdavidson/hate_speech_offensive")["train"].to_pandas().rename(columns={"class": "label"})
df_train, df_test = train_test_split(crudo[["tweet", "label"]], test_size=0.2, stratify=crudo["label"], random_state=0)
if PRUEBA_RAPIDA:
    df_train, df_test = df_train.sample(3000, random_state=0), df_test.sample(800, random_state=0)
df_train, df_test = df_train.reset_index(drop=True), df_test.reset_index(drop=True)
ETIQUETAS = {0: "discurso de odio", 1: "lenguaje ofensivo", 2: "ninguno"}
print(len(df_train), "tweets para entrenar y", len(df_test), "para probar")

In [ ]:
def matriz_confusion(y_real, y_pred, titulo="Matriz de confusión"):
    from sklearn.metrics import ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    ConfusionMatrixDisplay.from_predictions(y_real, y_pred, cmap="Blues", colorbar=False)
    plt.title(titulo)
    plt.show()

## Parte 1 · Conocer los datos

In [ ]:
df_total = pd.concat([df_train, df_test], ignore_index=True)
conteo = df_total["label"].value_counts().sort_index()
print(conteo)
conteo.rename(ETIQUETAS).plot(kind="bar", title="Tweets por clase");

*Respuesta:* la clase 1 (lenguaje ofensivo) supera por mucho a las otras dos: el conjunto **no está equilibrado**.

In [ ]:
df_total["largo"] = df_total["tweet"].str.split().str.len()
plt.figure(figsize=(8, 4))
sns.histplot(data=df_total, x="largo", hue=df_total["label"].map(ETIQUETAS), bins=30, kde=True)
plt.xlabel("Número de palabras"); plt.ylabel("Cantidad de tweets"); plt.title("Largo de los tweets por clase")
plt.show()
df_total.groupby("label")["largo"].mean().rename(ETIQUETAS).round(1)

*Respuesta:* los tweets de unas 10 palabras son los más abundantes. Los ofensivos y de odio tienden a ser más cortos
y directos; la clase «ninguno» es algo más larga en promedio.

In [ ]:
import re, html
from collections import Counter

def limpiar_texto(texto):
    texto = html.unescape(texto)                          # 1) &amp; → &
    texto = texto.lower()                                 # 2) minúsculas
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)       # 3) fuera los enlaces
    texto = re.sub(r"@\w+", "", texto)                    # 4) fuera las menciones
    texto = re.sub(r"#(\w+)", r"\1", texto)                # 5) hashtags: se quita «#» y queda la palabra
    texto = re.sub(r"[^a-z\s]", " ", texto)               # 6) solo letras y espacios
    return re.sub(r"\s+", " ", texto).strip()            # 7) espacios múltiples

def palabras_frecuentes(df, etiqueta, n=40):
    texto = df[df["label"] == etiqueta]["tweet"].apply(limpiar_texto)
    return Counter(" ".join(texto).split()).most_common(n)

In [ ]:
for etiqueta, nombre in ETIQUETAS.items():
    print(f"── {nombre}:", [p for p, _ in palabras_frecuentes(df_total, etiqueta)])

In [ ]:
def top_palabras(df, etiqueta, n=40):
    """Lo mismo con scikit-learn: además quita las palabras vacías del inglés (stop words)."""
    vectorizador = CountVectorizer(stop_words="english", max_features=n)
    X = vectorizador.fit_transform(df[df["label"] == etiqueta]["tweet"])
    return pd.DataFrame({"palabra": vectorizador.get_feature_names_out(), "frecuencia": X.sum(axis=0).A1}) \
             .sort_values("frecuencia", ascending=False)

top_palabras(df_total, 0).head(15)

*Respuesta:* las dos maneras dan resultados parecidos. Con Python puro se controla exactamente qué se filtra;
`CountVectorizer` trae filtros listos, como las palabras vacías, aunque deja pasar cadenas numéricas. En el fondo,
las dos clases ofensivas concentran **insultos y términos racistas** en inglés, repetitivos, que casi no aparecen en
la clase «ninguno».

## Parte 2 · Preprocesamiento

In [ ]:
df_train["tweet_limpio"] = df_train["tweet"].apply(limpiar_texto)
df_test["tweet_limpio"] = df_test["tweet"].apply(limpiar_texto)
df_train[["tweet", "tweet_limpio"]].head()

*Respuesta:* el preprocesamiento reduce ruido (enlaces, menciones, variaciones de forma) y **conserva los términos
ofensivos**, porque son justamente los que distinguen las clases. Para RoBERTa se usa el texto **original**: su
tokenizador ya maneja mayúsculas y puntuación, y le sirven como contexto.

## Parte 3 · BOW + regresión logística

In [ ]:
bow = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
X_train_bow = bow.fit_transform(df_train["tweet_limpio"])
X_test_bow = bow.transform(df_test["tweet_limpio"])
y_train, y_test = df_train["label"], df_test["label"]
print("Tamaño del vocabulario:", len(bow.vocabulary_))

clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train_bow, y_train)
y_pred = clf.predict(X_test_bow)
print(classification_report(y_test, y_pred, target_names=ETIQUETAS.values()))
matriz_confusion(y_test, y_pred, "BOW")

*Respuesta:* BOW anda bien en la clase mayoritaria, pero le cuesta el «discurso de odio»: contar palabras no basta
para separar una clase minoritaria que comparte vocabulario con el lenguaje ofensivo.

## Parte 4 · TF-IDF + regresión logística

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(df_train["tweet_limpio"])
X_test_tfidf = tfidf.transform(df_test["tweet_limpio"])
print("Tamaño del vocabulario:", len(tfidf.vocabulary_))

clf_tfidf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train_tfidf, y_train)
y_pred_tfidf = clf_tfidf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred_tfidf, target_names=ETIQUETAS.values()))
matriz_confusion(y_test, y_pred_tfidf, "TF-IDF")

*Respuesta:* TF-IDF mejora levemente a BOW, al bajar el peso de los términos muy frecuentes y resaltar los más
distintivos. El desequilibrio sigue afectando la detección del discurso de odio.

## Parte 5 · RoBERTa con ajuste fino (GPU)

In [ ]:
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          TrainingArguments, Trainer, set_seed)

set_seed(0)
ds = DatasetDict(train=Dataset.from_pandas(df_train[["tweet", "label"]]), test=Dataset.from_pandas(df_test[["tweet", "label"]]))
id2label = {0: "hate_speech", 1: "offensive_language", 2: "neither"}
label2id = {v: k for k, v in id2label.items()}
BASE = "FacebookAI/roberta-base"
tokenizador = AutoTokenizer.from_pretrained(BASE)
tokenizador.model_max_length = 130
ds_tok = ds.map(lambda e: tokenizador(e["tweet"], truncation="longest_first"))

f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    predicciones, etiquetas = eval_pred
    return f1.compute(references=etiquetas, predictions=np.argmax(predicciones, axis=1), average="micro")

In [ ]:
modelo = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=3, id2label=id2label, label2id=label2id)
argumentos = TrainingArguments(
    report_to="none", output_dir="clf_odio",
    eval_strategy="steps", save_strategy="steps", eval_steps=100, save_steps=100,
    learning_rate=2e-5, warmup_steps=250, weight_decay=0.18,
    per_device_train_batch_size=32, per_device_eval_batch_size=32, num_train_epochs=3,
    load_best_model_at_end=True, metric_for_best_model="f1", save_total_limit=2,
    fp16=torch.cuda.is_available(),
)
entrenador = Trainer(model=modelo, args=argumentos, data_collator=DataCollatorWithPadding(tokenizador, padding="longest"),
                     train_dataset=ds_tok["train"], eval_dataset=ds_tok["test"], compute_metrics=compute_metrics)
entrenador.train()
print(entrenador.evaluate(ds_tok["test"]))
MODELO_DIR = CARPETA / "roberta-odio"
entrenador.save_model(MODELO_DIR)          # guardado en Drive: no hay que repetir los ~20 minutos

In [ ]:
from tqdm.auto import tqdm

modelo_odio = AutoModelForSequenceClassification.from_pretrained(MODELO_DIR).to(DISPOSITIVO).eval()

@torch.no_grad()
def predecir_roberta(texto):
    entrada = tokenizador(str(texto), truncation=True, max_length=130, return_tensors="pt").to(DISPOSITIVO)
    return int(modelo_odio(**entrada).logits.argmax(-1).item())

df_test["pred_roberta"] = [predecir_roberta(t) for t in tqdm(df_test["tweet"])]
print(classification_report(df_test["label"], df_test["pred_roberta"], target_names=ETIQUETAS.values()))

*Respuesta:* RoBERTa supera a BOW y TF-IDF al modelar el **contexto**: mejores métricas globales y por clase, y una
mejor separación entre lenguaje ofensivo y discurso de odio.

## Parte 6 · Un modelo de lenguaje sin ejemplos (*zero-shot*)

El autor no obtuvo acceso a Llama-3.2-3B a tiempo y usó otro modelo decodificador (gemma-2b). Aquí se deja una
variable para elegir: por defecto un modelo abierto, sin trámite.

In [ ]:
SEMILLA = 25
muestra = df_test.groupby("label").sample(n=10, random_state=SEMILLA).reset_index(drop=True)
print(muestra["label"].value_counts().sort_index())

def instruccion(tweet):
    return f"""Eres un clasificador estricto de texto.

Clasifica cada tweet en exactamente una etiqueta:
0 = discurso de odio (ataque o deshumanización hacia un grupo protegido por identidad)
1 = lenguaje ofensivo (insultos o groserías, pero NO discurso de odio)
2 = ninguno (no ofensivo ni discurso de odio)

Reglas:
- Responde SOLO con un número: 0, 1 o 2.
- No expliques la respuesta.

Tweet:
\"\"\"{tweet}\"\"\"
Label:"""

def extraer_etiqueta(texto):
    hallados = re.findall(r"\b[012]\b", texto)
    return int(hallados[-1]) if hallados else 2    # sin respuesta válida: «ninguno» (criterio conservador)

*Diseño de la instrucción:* define las tres clases con el criterio de la anotación original (el odio ataca a un
**grupo** por su identidad; lo ofensivo insulta sin eso), obliga a responder un solo número y prohíbe explicar, para
que la respuesta sea fácil de leer automáticamente.

In [ ]:
MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"   # abierto, sin trámite
# MODELO_LLM = "meta-llama/Llama-3.2-3B-Instruct"   # pide aceptar su licencia en Hugging Face y tu HF_TOKEN en Secretos
from transformers import AutoModelForCausalLM

llm_tok = AutoTokenizer.from_pretrained(MODELO_LLM)
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, torch_dtype=torch.float16 if DISPOSITIVO == "cuda" else torch.float32).to(DISPOSITIVO)

@torch.no_grad()
def predecir_llm(prompt):
    mensajes = [{"role": "user", "content": prompt}]
    entrada = llm_tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to(DISPOSITIVO)
    salida = llm.generate(entrada, max_new_tokens=5, do_sample=False)
    return extraer_etiqueta(llm_tok.decode(salida[0][entrada.shape[1]:], skip_special_tokens=True))

muestra["pred_llm"] = [predecir_llm(instruccion(t)) for t in muestra["tweet"]]
print(classification_report(muestra["label"], muestra["pred_llm"], target_names=ETIQUETAS.values(), zero_division=0))

*Respuesta (con gemma-2b):* el *zero-shot* mostró un fuerte sesgo hacia «ninguno» y bajo desempeño en las clases
ofensivas: el modelo es conservador y no fue entrenado para esta tarea.

## Parte 7 · GPT sin ejemplos (opcional, con costo)

In [ ]:
def cliente_openai():
    """La clave va en el panel 🔑 Secretos como OPENAI_API_KEY. Sin clave, esta parte se salta."""
    if not EN_COLAB:
        return None
    from google.colab import userdata
    try:
        clave = userdata.get("OPENAI_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        print("Sin OPENAI_API_KEY en Secretos: se salta la parte con GPT.")
        return None
    from openai import OpenAI
    return OpenAI(api_key=clave)

cliente = cliente_openai()

def predecir_gpt(prompt):
    respuesta = cliente.chat.completions.create(
        model="gpt-4o-mini", max_tokens=5, temperature=0,
        messages=[{"role": "system", "content": "Eres un asistente de clasificación de texto."},
                  {"role": "user", "content": prompt}])
    return extraer_etiqueta(respuesta.choices[0].message.content)

if cliente:
    muestra["pred_gpt"] = [predecir_gpt(instruccion(t)) for t in muestra["tweet"]]
    print(classification_report(muestra["label"], muestra["pred_gpt"], target_names=ETIQUETAS.values(), zero_division=0))

*Respuesta:* GPT sin ejemplos equilibra mejor las clases que el modelo pequeño: muestra más comprensión semántica.

## Parte 8 · El modelo abierto con ejemplos (*few-shot*)

In [ ]:
K = 3   # ejemplos por clase
ejemplos = df_train.groupby("label").sample(n=K, random_state=7).reset_index(drop=True)

def instruccion_con_ejemplos(tweet):
    base = instruccion(tweet).split("Tweet:")[0].strip()
    lista = "".join(f'Tweet: """{f.tweet}"""\nLabel: {f.label}\n' for f in ejemplos.itertuples())
    return f'{base}\n\nEjemplos:\n{lista}\nAhora clasifica este tweet:\nTweet: """{tweet}"""\nLabel:'

print(instruccion_con_ejemplos("ejemplo")[-400:])

In [ ]:
muestra["pred_llm_ej"] = [predecir_llm(instruccion_con_ejemplos(t)) for t in muestra["tweet"]]
print(classification_report(muestra["label"], muestra["pred_llm_ej"], target_names=ETIQUETAS.values(), zero_division=0))

*Criterio de los ejemplos:* 3 por clase, al azar con semilla fija, para no sesgar hacia ninguna clase.
*Respuesta:* con ejemplos se reduce el sesgo del *zero-shot* y mejora sobre todo la detección del lenguaje ofensivo:
los ejemplos le muestran al modelo el criterio de decisión.

## Parte 9 · GPT con ejemplos (opcional, con costo)

In [ ]:
if cliente:
    muestra["pred_gpt_ej"] = [predecir_gpt(instruccion_con_ejemplos(t)) for t in muestra["tweet"]]
    print(classification_report(muestra["label"], muestra["pred_gpt_ej"], target_names=ETIQUETAS.values(), zero_division=0))

*Respuesta:* GPT con ejemplos obtuvo el mejor *recall* de los enfoques generativos. Con solo 30 tweets, las métricas
varían bastante de una muestra a otra.

## Parte 10 · Los clasificadores entrenados en la misma muestra

In [ ]:
muestra["tweet_limpio"] = muestra["tweet"].apply(limpiar_texto)
muestra["pred_bow"] = clf.predict(bow.transform(muestra["tweet_limpio"]))
muestra["pred_tfidf"] = clf_tfidf.predict(tfidf.transform(muestra["tweet_limpio"]))
for columna in ["pred_bow", "pred_tfidf"]:
    print(f"── {columna}\n", classification_report(muestra["label"], muestra[columna], target_names=ETIQUETAS.values(), zero_division=0))

In [ ]:
muestra["pred_roberta"] = [predecir_roberta(t) for t in muestra["tweet"]]
print(classification_report(muestra["label"], muestra["pred_roberta"], target_names=ETIQUETAS.values(), zero_division=0))

*Respuesta:* RoBERTa supera a BOW y TF-IDF también en la muestra: captura contexto y relaciones semánticas y
equilibra mejor las clases.

## Parte 11 · Conclusiones

In [ ]:
from sklearn.metrics import f1_score
columnas = [c for c in ["pred_bow", "pred_tfidf", "pred_roberta", "pred_llm", "pred_gpt", "pred_llm_ej", "pred_gpt_ej"] if c in muestra]
pd.Series({c.removeprefix("pred_"): f1_score(muestra["label"], muestra[c], average="macro") for c in columnas},
          name="F1 macro (muestra de 30)").round(3)

*Respuesta:* los modelos **supervisados** superan a los enfoques sin entrenar, sobre todo en las clases minoritarias.
Los modelos de lenguaje mejoran con ejemplos (*few-shot*), pero RoBERTa ajustado da el mejor desempeño global. A
favor de los modelos de lenguaje: no necesitan datos etiquetados ni entrenamiento. En contra: son más lentos o
cuestan por consulta, y varían con la instrucción.

**Fuentes:** las del [enunciado del proyecto](https://colab.research.google.com/github/psalgadm7/miatlas-cuadernos/blob/main/cuadernos/nlp-detector-de-odio.ipynb).